# 03 - Routing Basics: if/else and Method Branching

In this notebook you will:
- Learn Kamailio cfg control flow
- Use `is_method()` to handle different message types
- Understand `route[]` blocks

---

## Kamailio cfg Structure

A Kamailio cfg file has three sections:

```kamailio
# 1. Global Parameters
listen=udp:10.0.0.1:5060

# 2. Modules
loadmodule "sl.so"
loadmodule "tm.so"

# 3. Routing Logic
request_route {
    if (is_method("REGISTER")) {
        save("location");
    }
    if (is_method("INVITE")) {
        record_route();
        t_relay();
    }
}
```

In this notebook we practice the **Routing Logic** section.

## 1. Basic if/else

Kamailio's if/else is similar to C.

In [ ]:
%%sip INVITE
From: <sip:1001@example.com>;tag=test1
To: <sip:1002@example.com>

In [ ]:
# Check the method
if (is_method("INVITE")) {
    xlog("This is an INVITE request");
}

In [ ]:
# This should be FALSE
if (is_method("REGISTER")) {
    xlog("This won't be printed");
}

## 2. if/else Branching

You can handle different methods with different logic.

In [ ]:
$var(method) = "INVITE";

if ($var(method) == "INVITE") {
    xlog("Processing INVITE");
} else {
    xlog("Not INVITE");
}

## 3. has_totag() — In-Dialog vs Initial Requests

`has_totag()` checks if the To header has a tag parameter.
- **Initial requests** (INVITE, REGISTER): no tag → FALSE
- **In-dialog requests** (BYE, re-INVITE): has tag → TRUE

This matters because in-dialog requests need different handling.

In [ ]:
# Initial INVITE: no To tag
%%sip INVITE
From: <sip:1001@example.com>;tag=from1
To: <sip:1002@example.com>

In [ ]:
# Initial INVITE, so has_totag() → FALSE
if (has_totag()) {
    xlog("In-dialog request");
} else {
    xlog("Initial request — needs routing");
}

## 4. Negation Operator (!)

Use `!` to negate a condition.

In [ ]:
# Not a REGISTER (it's INVITE, so TRUE)
if (!is_method("REGISTER")) {
    xlog("Not a REGISTER — proceed with routing");
}

## 5. route() Calls

In Kamailio, `route(name)` calls another routing block — a way to modularize code.

In [ ]:
if (is_method("REGISTER")) {
    route(REGISTRAR);
} else {
    route(LOCATION);
}

## 6. drop / exit

- `drop`: Immediately stop and discard the message
- `exit`: Stop processing (similar to drop)
- `return`: Exit the current route block only

In [ ]:
$var(blocked) = 1;

if ($var(blocked) == 1) {
    xlog("Blocked user — dropping message");
    drop;
} else {
    xlog("Allowed");
}

## 7. send_reply — Sending Responses

You can send SIP response codes directly.

| Code | Meaning |
|------|---------|
| 200 | OK |
| 401 | Unauthorized |
| 403 | Forbidden |
| 404 | Not Found |
| 503 | Service Unavailable |

In [ ]:
$var(user_exists) = 0;

if ($var(user_exists) == 0) {
    send_reply(404, "Not Found");
} else {
    send_reply(200, "OK");
}

---

### Summary

| Concept | Syntax |
|---------|--------|
| Conditional | `if (condition) { ... } else { ... }` |
| Method check | `is_method("INVITE")` |
| To tag check | `has_totag()` |
| Negation | `!condition` |
| Comparison | `==`, `!=`, `>`, `<` |
| Route block call | `route(NAME);` |
| Discard message | `drop;` or `exit;` |
| Send response | `send_reply(code, "reason");` |

Next: **Intermediate/01-transformations.ipynb** →